# Lab 4: Apache Spark 3.5+ Foundations & DataFrame API
**Z5008 Big Data Lab | IIT Madras Zanzibar | Dr. Innocent Nyalala** | 
**Date:** Monday 30 March 2026

## Learning Objectives
- Understand Spark's lazy evaluation model
- Master multiple data formats (CSV, JSON, Parquet, Delta)
- Apply complex transformations (explode, pivot, window functions)
- Compare Python UDFs vs Pandas UDFs vs built-in functions
- Choose between broadcast and sort-merge joins
- Benchmark caching strategies
- Read physical execution plans


## Part 1: SparkSession Setup

In [1]:
import os, time
import urllib.request
import pandas as pd
import numpy as np

# ── Kill any stale SparkContext
try:
    from pyspark import SparkContext
    _ctx = SparkContext._active_spark_context
    if _ctx is not None:
        print('[INIT] Stale SparkContext — stopping.')
        _ctx.stop()
        time.sleep(2)
except Exception:
    pass

# ── Download ALL required JARs (Delta + AWS S3A — same as Lab 3)
print('[1/3] Checking JARs...')
JARS = {
    'delta-spark_2.12-3.2.0.jar':
        'https://repo1.maven.org/maven2/io/delta/delta-spark_2.12/3.2.0/delta-spark_2.12-3.2.0.jar',
    'delta-storage-3.2.0.jar':
        'https://repo1.maven.org/maven2/io/delta/delta-storage/3.2.0/delta-storage-3.2.0.jar',
    'hadoop-aws-3.3.4.jar':
        'https://repo1.maven.org/maven2/org/apache/hadoop/hadoop-aws/3.3.4/hadoop-aws-3.3.4.jar',
    'aws-java-sdk-bundle-1.12.262.jar':
        'https://repo1.maven.org/maven2/com/amazonaws/aws-java-sdk-bundle/1.12.262/aws-java-sdk-bundle-1.12.262.jar',
}
jar_paths = []
for fname, url in JARS.items():
    fpath = os.path.join(os.getcwd(), fname)
    jar_paths.append(fpath)
    if os.path.exists(fpath):
        print(f'      {fname} already present.')
    else:
        print(f'      Downloading {fname}...')
        urllib.request.urlretrieve(url, fpath)
        print(f'      {fname} done.')
local_jars = ','.join(jar_paths)

# ── Start SparkSession
print('[2/3] Starting SparkSession...')
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
from pyspark import StorageLevel

spark = (
    SparkSession.builder
    .appName('Lab04-SparkFoundations')
    .master('local[*]')
    .config('spark.jars', local_jars)
    .config('spark.sql.extensions',
            'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog',
            'org.apache.spark.sql.delta.catalog.DeltaCatalog')
    .config('spark.sql.shuffle.partitions',                  '20')
    .config('spark.sql.adaptive.enabled',                    'true')
    .config('spark.sql.adaptive.coalescePartitions.enabled', 'true')
    .config('spark.sql.autoBroadcastJoinThreshold',          '10m')
    .config('spark.hadoop.fs.s3a.endpoint',          'http://minio:9000')
    .config('spark.ui.port', '4040')
    .config('spark.hadoop.fs.s3a.access.key',        'admin')
    .config('spark.hadoop.fs.s3a.secret.key',        'bigdata123')
    .config('spark.hadoop.fs.s3a.path.style.access', 'true')
    .config('spark.hadoop.fs.s3a.impl',
            'org.apache.hadoop.fs.s3a.S3AFileSystem')
    .config('spark.hadoop.fs.s3a.aws.credentials.provider',
            'org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider')
    .getOrCreate()
)

spark.sparkContext.setLogLevel('WARN')
STORAGE_ROOT = 's3a://warehouse/lab04'

print('[3/3] Done.')
print('=' * 45)
print('Spark version:       ', spark.version)
print('Default parallelism: ', spark.sparkContext.defaultParallelism)
print('AQE enabled:         ', spark.conf.get('spark.sql.adaptive.enabled'))
print('Storage root:        ', STORAGE_ROOT)
print('=' * 45)

[1/3] Checking JARs...
      delta-spark_2.12-3.2.0.jar done.
      delta-storage-3.2.0.jar done.
      hadoop-aws-3.3.4.jar already present.
      aws-java-sdk-bundle-1.12.262.jar already present.
[2/3] Starting SparkSession...
[3/3] Done.
Spark version:        3.5.0
Default parallelism:  4
AQE enabled:          true
Storage root:         s3a://warehouse/lab04


In [2]:
print('Spark UI available at:', spark.sparkContext.uiWebUrl)

Spark UI available at: http://930ff5aed190:4041


## Part 2: Generate E-Commerce Orders Dataset (100K rows)

In [3]:
np.random.seed(42)
n = 100_000
countries  = ['Kenya','Tanzania','Uganda','Ghana','Nigeria','Ethiopia']
categories = ['Electronics','Furniture','Clothing','Food','Books','Health']
statuses   = ['completed','pending','cancelled','refunded']
methods    = ['M-Pesa','Card','Bank Transfer','Cash on Delivery']

records = {
    'order_id':       [f'ORD{i:07d}' for i in range(n)],
    'customer_id':    [f'CUS{np.random.randint(1,10001):05d}' for _ in range(n)],
    'country':        np.random.choice(countries, n, p=[0.35,0.25,0.15,0.10,0.10,0.05]).astype(str).tolist(),
    'category':       np.random.choice(categories, n).astype(str).tolist(),
    'quantity':       np.random.randint(1, 11, n).tolist(),
    'unit_price':     np.round(np.random.uniform(5.0, 500.0, n), 2).tolist(),
    'order_date':     [pd.Timestamp('2025-01-01') + pd.Timedelta(days=int(d))
                       for d in np.random.randint(0, 365, n)],
    'status':         np.random.choice(statuses, n, p=[0.70,0.15,0.10,0.05]).astype(str).tolist(),
    'payment_method': np.random.choice(methods, n).astype(str).tolist(),
}

df_pd = pd.DataFrame(records)
df_pd['order_total'] = (df_pd['quantity'] * df_pd['unit_price']).round(2)

df_orders = spark.createDataFrame(df_pd)
df_orders.printSchema()
df_orders.show(5)
print(f'Total rows: {df_orders.count():,}')

df_orders.write.mode('overwrite').parquet(f'{STORAGE_ROOT}/orders_parquet/')
df_orders.write.format('delta').mode('overwrite').save(f'{STORAGE_ROOT}/orders_delta/')
df_orders.createOrReplaceTempView('orders')
print(f'Saved to MinIO. Check localhost:9001 -> warehouse/lab04/')

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- country: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: long (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- order_date: timestamp (nullable = true)
 |-- status: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_total: double (nullable = true)

+----------+-----------+--------+-----------+--------+----------+-------------------+---------+--------------+-----------+
|  order_id|customer_id| country|   category|quantity|unit_price|         order_date|   status|payment_method|order_total|
+----------+-----------+--------+-----------+--------+----------+-------------------+---------+--------------+-----------+
|ORD0000000|   CUS07271|   Kenya|       Food|       7|     90.85|2025-03-03 00:00:00|completed| Bank Transfer|     635.95|
|ORD0000001|   CUS00861|   Kenya|       Food|       7|    480.61|2025-04-15 00:00:00|com

## Part 2b: RDD vs DataFrame — The Evolution

In [4]:
# ============================================================
# Part 2b: RDD vs DataFrame — The Evolution
# ============================================================

import time

# Use Python built-in round explicitly to avoid conflict with pyspark round()
_round = __builtins__['round'] if isinstance(__builtins__, dict) else __builtins__.round

print('=' * 55)
print('APPROACH 1: RDD (low-level, no Catalyst optimizer)')
print('=' * 55)

rdd = spark.sparkContext.parallelize(df_pd.to_dict('records'))

t0 = time.time()
rdd_result = (
    rdd
    .filter(lambda r: r['status'] == 'completed')
    .filter(lambda r: r['country'] == 'Kenya')
    .map(lambda r: (r['category'], (r['order_total'], 1)))
    .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
    .map(lambda x: (x[0], _round(x[1][0] / x[1][1], 2)))
    .collect()
)
t_rdd = time.time() - t0

print('RDD result (Kenya completed orders — avg by category):')
for cat, avg_val in sorted(rdd_result):
    print(f'  {cat:15s}  avg = {avg_val}')
print(f'RDD time: {t_rdd:.2f}s')
print('RDD has NO query plan — Python executes row by row')
print()

# ── DataFrame approach
print('=' * 55)
print('APPROACH 2: DataFrame (Catalyst optimizer + Tungsten)')
print('=' * 55)

t0 = time.time()
df_result = (
    df_orders
    .filter((col('status') == 'completed') & (col('country') == 'Kenya'))
    .groupBy('category')
    .agg(round(avg('order_total'), 2).alias('avg_order'))
    .orderBy('category')
)
df_result.show()
t_df = time.time() - t0

print(f'DataFrame time: {t_df:.2f}s')
print(f'Speedup over RDD: {t_rdd / t_df:.1f}x')
print()
print('KEY INSIGHT: Same result. DataFrame is faster because:')
print('  1. Catalyst rewrites the query plan before execution')
print('  2. Tungsten uses off-heap memory and code generation')
print('  3. Predicate pushdown filters at file read time')
print()
print('RULE: Never use RDDs for tabular data. Always use DataFrames.')

APPROACH 1: RDD (low-level, no Catalyst optimizer)
RDD result (Kenya completed orders — avg by category):
  Books            avg = 1362.43
  Clothing         avg = 1401.85
  Electronics      avg = 1372.05
  Food             avg = 1376.81
  Furniture        avg = 1380.33
  Health           avg = 1392.73
RDD time: 1.38s
RDD has NO query plan — Python executes row by row

APPROACH 2: DataFrame (Catalyst optimizer + Tungsten)
+-----------+---------+
|   category|avg_order|
+-----------+---------+
|      Books|  1362.43|
|   Clothing|  1401.85|
|Electronics|  1372.05|
|       Food|  1376.81|
|  Furniture|  1380.33|
|     Health|  1392.73|
+-----------+---------+

DataFrame time: 0.83s
Speedup over RDD: 1.7x

KEY INSIGHT: Same result. DataFrame is faster because:
  1. Catalyst rewrites the query plan before execution
  2. Tungsten uses off-heap memory and code generation
  3. Predicate pushdown filters at file read time

RULE: Never use RDDs for tabular data. Always use DataFrames.


## Part 3: Lazy Evaluation — Transformations vs Actions

In [5]:
# TRANSFORMATIONS (lazy -- no computation yet)
df_completed = df_orders.filter(col('status') == 'completed')
df_selected  = df_completed.select('order_id','country','category','order_total','order_date')
df_grouped   = df_selected.groupBy('country','category') \
                          .agg(count('order_id').alias('n_orders'),
                               round(avg('order_total'), 2).alias('avg_total'),
                               round(sum('order_total'), 2).alias('total_revenue'))

print('Transformations defined -- check localhost:4040: no jobs yet!')

# ACTION triggers execution of the entire DAG
t0 = time.time()
df_grouped.show(10)
print(f'Action completed in {time.time()-t0:.2f}s -- check localhost:4040: 1 job appeared')

# Read the physical plan
print('\n=== Physical Plan ===')
df_grouped.explain(mode='formatted')


Transformations defined -- check localhost:4040: no jobs yet!
+--------+-----------+--------+---------+-------------+
| country|   category|n_orders|avg_total|total_revenue|
+--------+-----------+--------+---------+-------------+
|Ethiopia|Electronics|     597|  1395.49|    833109.27|
| Nigeria|      Books|    1131|  1442.62|   1631598.28|
|   Ghana|Electronics|    1154|  1412.83|    1630404.4|
|Tanzania|       Food|    2846|  1378.52|   3923257.76|
|   Kenya|  Furniture|    4021|  1380.33|   5550322.16|
|   Kenya|      Books|    3994|  1362.43|   5441547.81|
| Nigeria|   Clothing|    1232|   1403.7|   1729359.44|
|   Ghana|      Books|    1136|  1360.48|   1545503.05|
| Nigeria|     Health|    1152|  1476.63|   1701075.44|
|  Uganda|  Furniture|    1715|  1385.48|    2376106.4|
+--------+-----------+--------+---------+-------------+
only showing top 10 rows

Action completed in 0.48s -- check localhost:4040: 1 job appeared

=== Physical Plan ===
== Physical Plan ==
AdaptiveSparkPlan (

## Part 3b: Spark UI Guided Tour

In [6]:
# ============================================================
# Open localhost:4040 in your browser NOW and follow along
# ============================================================

print('Spark UI:', spark.sparkContext.uiWebUrl)
print()
print('STEP 1 — Open localhost:4040 in your browser')
print('         You should see the Spark Jobs page')
print()

# Trigger a job
print('Running a job now — watch localhost:4040 as it executes...')
print()

t0 = time.time()
ui_demo = (
    df_orders
    .filter(col('status') == 'completed')
    .groupBy('country', 'category')
    .agg(
        count('*').alias('n_orders'),
        round(sum('order_total'), 2).alias('revenue')
    )
    .orderBy(desc('revenue'))
)
ui_demo.show(10)
t_job = time.time() - t0
print(f'Job completed in {t_job:.2f}s')

print()
print('=' * 55)
print('NOW EXPLORE localhost:4040 — checklist:')
print('=' * 55)
print()
print('JOBS tab:')
print('  How many jobs ran for the .show(10) above?')
print('  Click a job to drill into its stages.')
print()
print('STAGES tab:')
print('  Look for "Shuffle Read" and "Shuffle Write" columns.')
print('  High shuffle = network bottleneck in your query.')
print()
print('SQL / DATAFRAME tab:')
print('  Click the most recent query.')
print('  Find the Exchange node — that is your shuffle.')
print('  Find Filter inside FileScan — predicate pushdown working.')
print()
print('STORAGE tab:')
print('  Empty now. Run the cache cell (Part 8) and return here.')
print('  You will see the cached DataFrame appear.')
print()
print('ENVIRONMENT tab:')
print('  Confirm spark.sql.adaptive.enabled = true')
print('  Confirm spark.sql.shuffle.partitions = 20')

print()
print('=' * 55)
# Show the explain plan alongside the UI
print('Physical plan for the query above (matches SQL tab):')
print('=' * 55)
ui_demo.explain(mode='formatted')

Spark UI: http://930ff5aed190:4041

STEP 1 — Open localhost:4040 in your browser
         You should see the Spark Jobs page

Running a job now — watch localhost:4040 as it executes...

+--------+-----------+--------+----------+
| country|   category|n_orders|   revenue|
+--------+-----------+--------+----------+
|   Kenya|   Clothing|    4164|5837303.85|
|   Kenya|     Health|    4120|5738036.39|
|   Kenya|       Food|    4094|5636674.51|
|   Kenya|Electronics|    4074|5589731.19|
|   Kenya|  Furniture|    4021|5550322.16|
|   Kenya|      Books|    3994|5441547.81|
|Tanzania|Electronics|    2999| 4129136.9|
|Tanzania|   Clothing|    3005|4092296.61|
|Tanzania|      Books|    2953|4070476.25|
|Tanzania|     Health|    2869|3953476.59|
+--------+-----------+--------+----------+
only showing top 10 rows

Job completed in 0.48s

NOW EXPLORE localhost:4040 — checklist:

JOBS tab:
  How many jobs ran for the .show(10) above?
  Click a job to drill into its stages.

STAGES tab:
  Look for "S

## Part 4: Explicit Schema vs inferSchema

In [7]:
# Define schema explicitly (single-pass read)
order_schema = StructType([
    StructField('order_id',       StringType(),    False),
    StructField('customer_id',    StringType(),    True),
    StructField('country',        StringType(),    True),
    StructField('category',       StringType(),    True),
    StructField('quantity',       IntegerType(),   True),
    StructField('unit_price',     DoubleType(),    True),
    StructField('order_date',     TimestampType(), True),
    StructField('status',         StringType(),    True),
    StructField('payment_method', StringType(),    True),
    StructField('order_total',    DoubleType(),    True),
])

# Read from Parquet (schema is embedded -- explicit schema not needed for Parquet)
df_typed = spark.read.parquet(f'{STORAGE_ROOT}/orders_parquet/')
print('Schema from Parquet (embedded):')
df_typed.printSchema()
print(f'Row count: {df_typed.count():,}')
df_typed.createOrReplaceTempView('orders_typed')


Schema from Parquet (embedded):
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- country: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: long (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- order_date: timestamp (nullable = true)
 |-- status: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_total: double (nullable = true)

Row count: 100,000


## Part 5: Complex Transforms — Window Functions, Pivot, Explode

In [8]:
# --- WINDOW: top 5 orders per country by value ---
window_country = Window.partitionBy('country').orderBy(desc('order_total'))
df_top5 = df_orders \
    .withColumn('rank_in_country', rank().over(window_country)) \
    .filter(col('rank_in_country') <= 5) \
    .select('country','order_id','order_total','rank_in_country')
print('=== Top 5 Orders per Country ===')
df_top5.show(15)

# --- WINDOW: month-over-month running total ---
window_run = Window.partitionBy('country').orderBy('month') \
                   .rowsBetween(Window.unboundedPreceding, Window.currentRow)
df_monthly = df_orders.withColumn('month', date_format('order_date','yyyy-MM')) \
    .groupBy('country','month').agg(sum('order_total').alias('monthly_rev'))
df_monthly.withColumn('running_total',
    round(sum('monthly_rev').over(window_run), 2)) \
    .orderBy('country','month').show(20)

# --- PIVOT: orders per category per country ---
print('=== Pivot: Orders by Category x Country ===')
df_orders.groupBy('category') \
    .pivot('country', ['Kenya','Tanzania','Uganda','Ghana','Nigeria','Ethiopia']) \
    .agg(count('order_id')) \
    .show()

# --- EXPLODE: split tags array into rows ---
df_tags = df_orders \
    .withColumn('tags', array(col('status'), col('payment_method'), col('category'))) \
    .withColumn('tag', explode(col('tags'))) \
    .groupBy('tag').count().orderBy(desc('count'))
print('=== Exploded Tags Frequency ===')
df_tags.show(15)


=== Top 5 Orders per Country ===
+--------+----------+-----------+---------------+
| country|  order_id|order_total|rank_in_country|
+--------+----------+-----------+---------------+
|Ethiopia|ORD0059119|     4978.7|              1|
|Ethiopia|ORD0088576|     4966.6|              2|
|Ethiopia|ORD0002832|     4962.3|              3|
|Ethiopia|ORD0050618|     4961.5|              4|
|Ethiopia|ORD0050240|     4958.7|              5|
|   Ghana|ORD0085553|     4995.0|              1|
|   Ghana|ORD0023571|     4991.3|              2|
|   Ghana|ORD0019089|     4988.7|              3|
|   Ghana|ORD0049302|     4979.4|              4|
|   Ghana|ORD0067171|     4976.2|              5|
|   Kenya|ORD0060287|     4999.3|              1|
|   Kenya|ORD0083599|     4999.3|              1|
|   Kenya|ORD0090102|     4999.3|              1|
|   Kenya|ORD0085744|     4995.0|              4|
|   Kenya|ORD0094395|     4993.6|              5|
+--------+----------+-----------+---------------+
only showing top 

## Part 6: Python UDFs vs Pandas UDFs vs Built-in Functions

In [9]:
# Python UDF (slow: row-by-row)
@udf(returnType=StringType())
def order_tier_udf(total):
    if total is None:    return 'Unknown'
    if total >= 1000.0:  return 'High-Value'
    if total >= 200.0:   return 'Standard'
    return 'Low-Value'

t0 = time.time()
df_orders.withColumn('tier', order_tier_udf(col('order_total'))).count()
t_udf = time.time() - t0
print(f'Python UDF:      {t_udf:.2f}s')

# Pandas UDF (faster: vectorised batches via Apache Arrow)
try:
    import pyarrow  # noqa: F401

    @pandas_udf(StringType())
    def order_tier_pandas(series):
        return series.apply(
            lambda x: 'High-Value' if x >= 1000 else ('Standard' if x >= 200 else 'Low-Value')
        )

    t0 = time.time()
    df_orders.withColumn('tier', order_tier_pandas(col('order_total'))).count()
    t_pandas = time.time() - t0
    print(f'Pandas UDF:      {t_pandas:.2f}s  ({t_udf/t_pandas:.1f}x faster than Python UDF)')

except ImportError:
    t_pandas = t_udf  # fallback so the ratio below still prints
    print('Pandas UDF skipped: pyarrow not installed. Run: pip install pyarrow')

# Built-in Spark function (fastest: pure JVM)
t0 = time.time()
df_orders.withColumn('tier',
    when(col('order_total') >= 1000, 'High-Value')
    .when(col('order_total') >= 200, 'Standard')
    .otherwise('Low-Value')).count()
t_builtin = time.time() - t0
print(f'Built-in when(): {t_builtin:.2f}s  ({t_udf/t_builtin:.1f}x faster than Python UDF)')
print('\nConclusion: Always use built-in functions when available!')


Python UDF:      0.31s
Pandas UDF:      0.35s  (0.9x faster than Python UDF)
Built-in when(): 0.32s  (1.0x faster than Python UDF)

Conclusion: Always use built-in functions when available!


## Part 7: Join Strategies — Broadcast vs Sort-Merge

In [10]:
# Small lookup table (6 rows)
country_meta = spark.createDataFrame([
    ('Kenya','East Africa','EAT','+3'),
    ('Tanzania','East Africa','EAT','+3'),
    ('Uganda','East Africa','EAT','+3'),
    ('Ghana','West Africa','GMT','0'),
    ('Nigeria','West Africa','WAT','+1'),
    ('Ethiopia','East Africa','EAT','+3'),
], ['country','region','tz_abbr','utc_offset'])

# Sort-Merge Join (default)
print('=== Sort-Merge Join Plan ===')
df_orders.join(country_meta, 'country').explain(mode='simple')

t0 = time.time()
smj_count = df_orders.join(country_meta, 'country').count()
t_smj = time.time() - t0
print(f'SortMerge Join: {t_smj:.2f}s, rows={smj_count:,}')

# Broadcast Join
print('\n=== Broadcast Join Plan ===')
df_orders.join(broadcast(country_meta), 'country').explain(mode='simple')

t0 = time.time()
bcast_count = df_orders.join(broadcast(country_meta), 'country').count()
t_bcast = time.time() - t0
print(f'Broadcast Join: {t_bcast:.2f}s, rows={bcast_count:,}')
print(f'Speedup: {t_smj/t_bcast:.1f}x')

# Save enriched dataset
df_enriched = df_orders.join(broadcast(country_meta), 'country')
df_enriched.createOrReplaceTempView('orders_enriched')


=== Sort-Merge Join Plan ===
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [country#2, order_id#0, customer_id#1, category#3, quantity#4L, unit_price#5, order_date#6, status#7, payment_method#8, order_total#9, region#1209, tz_abbr#1210, utc_offset#1211]
   +- SortMergeJoin [country#2], [country#1208], Inner
      :- Sort [country#2 ASC NULLS FIRST], false, 0
      :  +- Exchange hashpartitioning(country#2, 20), ENSURE_REQUIREMENTS, [plan_id=853]
      :     +- Filter isnotnull(country#2)
      :        +- Scan ExistingRDD[order_id#0,customer_id#1,country#2,category#3,quantity#4L,unit_price#5,order_date#6,status#7,payment_method#8,order_total#9]
      +- Sort [country#1208 ASC NULLS FIRST], false, 0
         +- Exchange hashpartitioning(country#1208, 20), ENSURE_REQUIREMENTS, [plan_id=854]
            +- Filter isnotnull(country#1208)
               +- Scan ExistingRDD[country#1208,region#1209,tz_abbr#1210,utc_offset#1211]


SortMerge Join: 0.67s, rows=100,000

=== 

## Part 8: Caching Benchmark

In [11]:
df_base = df_orders.filter(col('status') == 'completed') \
                   .join(broadcast(country_meta), 'country') \
                   .withColumn('month', date_format('order_date','yyyy-MM'))

# Without cache
t0 = time.time()
df_base.groupBy('country','category').agg(count('*').alias('n')).collect()
df_base.groupBy('region','month').agg(sum('order_total').alias('rev')).collect()
df_base.groupBy('payment_method').agg(avg('order_total').alias('avg_val')).collect()
t_no_cache = time.time() - t0
print(f'No cache: {t_no_cache:.2f}s  (3 full scans from storage)')

# With cache
df_cached = df_base.cache()
df_cached.count()  # materialise
print('Cache materialised. Check Spark UI -> Storage tab.')

t0 = time.time()
df_cached.groupBy('country','category').agg(count('*').alias('n')).collect()
df_cached.groupBy('region','month').agg(sum('order_total').alias('rev')).collect()
df_cached.groupBy('payment_method').agg(avg('order_total').alias('avg_val')).collect()
t_cache = time.time() - t0
print(f'With cache: {t_cache:.2f}s  ({t_no_cache/t_cache:.1f}x speedup)')

df_cached.unpersist()
print('Cache freed. Check Storage tab: entry should be gone.')


No cache: 1.65s  (3 full scans from storage)
Cache materialised. Check Spark UI -> Storage tab.
With cache: 0.79s  (2.1x speedup)
Cache freed. Check Storage tab: entry should be gone.


## Part 9: Reading Execution Plans

In [12]:
# Query 1: Filter + GroupBy
q1 = df_orders.filter(col('country') == 'Kenya') \
              .groupBy('category') \
              .agg(count('*').alias('n'), avg('order_total').alias('avg_val'))
print('=== Query 1: Filter + GroupBy ===')
q1.explain(mode='formatted')

# Query 2: Join + GroupBy
q2 = df_orders.join(broadcast(country_meta), 'country') \
              .groupBy('region','category') \
              .agg(sum('order_total').alias('total'), count('*').alias('n')) \
              .orderBy(desc('total'))
print('\n=== Query 2: Join + GroupBy + OrderBy ===')
q2.explain(mode='formatted')

# Query 3: Window function
q3 = df_orders.withColumn('rank_in_cat',
        rank().over(Window.partitionBy('category').orderBy(desc('order_total')))) \
    .filter(col('rank_in_cat') <= 10)
print('\n=== Query 3: Window Function ===')
q3.explain(mode='formatted')


=== Query 1: Filter + GroupBy ===
== Physical Plan ==
AdaptiveSparkPlan (7)
+- HashAggregate (6)
   +- Exchange (5)
      +- HashAggregate (4)
         +- Project (3)
            +- Filter (2)
               +- Scan ExistingRDD (1)


(1) Scan ExistingRDD
Output [10]: [order_id#0, customer_id#1, country#2, category#3, quantity#4L, unit_price#5, order_date#6, status#7, payment_method#8, order_total#9]
Arguments: [order_id#0, customer_id#1, country#2, category#3, quantity#4L, unit_price#5, order_date#6, status#7, payment_method#8, order_total#9], MapPartitionsRDD[4] at applySchemaToPythonRDD at NativeMethodAccessorImpl.java:0, ExistingRDD, UnknownPartitioning(0)

(2) Filter
Input [10]: [order_id#0, customer_id#1, country#2, category#3, quantity#4L, unit_price#5, order_date#6, status#7, payment_method#8, order_total#9]
Condition : (isnotnull(country#2) AND (country#2 = Kenya))

(3) Project
Output [2]: [category#3, order_total#9]
Input [10]: [order_id#0, customer_id#1, country#2, category#3

## Part 10: Write Results to Multiple Formats

In [13]:
summary = df_orders.withColumn('month', date_format('order_date','yyyy-MM')) \
    .groupBy('month','country','category') \
    .agg(
        count('order_id').alias('n_orders'),
        round(sum('order_total'), 2).alias('total_revenue'),
        round(avg('order_total'), 2).alias('avg_order_value'),
        round(min('order_total'), 2).alias('min_order'),
        round(max('order_total'), 2).alias('max_order')
    )

# Write as Delta Lake
summary.write.format('delta').mode('overwrite').partitionBy('month') \
    .save(f'{STORAGE_ROOT}/gold/monthly_summary/')
print('Wrote Delta table. Reading back:')
spark.read.format('delta').load(f'{STORAGE_ROOT}/gold/monthly_summary/').show(5)

# Write as Parquet
summary.write.mode('overwrite').partitionBy('month') \
    .parquet(f'{STORAGE_ROOT}/gold/summary_parquet/')

# Write as single CSV for Excel-sharing
summary.coalesce(1).write.mode('overwrite').option('header','true') \
    .csv(f'{STORAGE_ROOT}/gold/summary_csv/')

print('All formats written. Verify in MinIO Console (localhost:9001) -> warehouse/lab04/gold/')
print('Spark UI still live at localhost:4040 — explore Jobs, Stages, Storage, SQL tabs.')
print('Call spark.stop() only when you are completely done.')

Wrote Delta table. Reading back:
+-------+--------+--------+--------+-------------+---------------+---------+---------+
|  month| country|category|n_orders|total_revenue|avg_order_value|min_order|max_order|
+-------+--------+--------+--------+-------------+---------------+---------+---------+
|2025-02|   Kenya|  Health|     442|    623459.25|        1410.54|    18.72|   4847.3|
|2025-02|Ethiopia|Clothing|      64|    101541.77|        1586.59|    20.67|   4556.8|
|2025-02| Nigeria|   Books|     125|    185165.61|        1481.32|    15.35|   4615.7|
|2025-02|Tanzania|   Books|     278|    393868.83|        1416.79|    19.05|   4942.7|
|2025-02|Ethiopia|    Food|      60|     94551.45|        1575.86|   122.36|   4322.3|
+-------+--------+--------+--------+-------------+---------------+---------+---------+
only showing top 5 rows

All formats written. Verify in MinIO Console (localhost:9001) -> warehouse/lab04/gold/
Spark UI still live at localhost:4040 — explore Jobs, Stages, Storage, S

## Part 11: Assignment 4 Scaffolding

In [14]:
# ============================================================
# Use this as your starting template for Assignment 4
# Replace the dataset path and adapt the operations
# ============================================================

print('Assignment 4 requires 8 operations on your own dataset.')
print('This cell demonstrates all 8 using the lab dataset.')
print('Adapt each section for your chosen dataset.')
print()

# ── Operation 1: Explicit schema (already done in Part 4)
print('[1/8] Explicit schema — read from MinIO with defined schema')
order_schema = StructType([
    StructField('order_id',       StringType(),    False),
    StructField('customer_id',    StringType(),    True),
    StructField('country',        StringType(),    True),
    StructField('category',       StringType(),    True),
    StructField('quantity',       LongType(),      True),
    StructField('unit_price',     DoubleType(),    True),
    StructField('order_date',     TimestampType(), True),
    StructField('status',         StringType(),    True),
    StructField('payment_method', StringType(),    True),
    StructField('order_total',    DoubleType(),    True),
])
df_schema = spark.read.schema(order_schema).parquet(f'{STORAGE_ROOT}/orders_parquet/')
print(f'   Rows loaded with explicit schema: {df_schema.count():,}')

# ── Operation 2: Filter
print('[2/8] Filter — completed orders over $500')
df_high = df_schema.filter(
    (col('status') == 'completed') & (col('order_total') > 500)
)
print(f'   High-value completed orders: {df_high.count():,}')

# ── Operation 3: GroupBy + Agg
print('[3/8] GroupBy + Aggregation')
df_schema.groupBy('country').agg(
    count('order_id').alias('n_orders'),
    round(sum('order_total'), 2).alias('total_revenue'),
    round(avg('order_total'), 2).alias('avg_order')
).orderBy(desc('total_revenue')).show()

# ── Operation 4: Window function
print('[4/8] Window function — rank orders by value within country')
win = Window.partitionBy('country').orderBy(desc('order_total'))
df_schema.withColumn('rank', rank().over(win)) \
    .filter(col('rank') <= 3) \
    .select('country', 'order_id', 'order_total', 'rank') \
    .orderBy('country', 'rank').show(12)

# ── Operation 5: Explode + Pivot
print('[5/8] Pivot — orders per payment method per country')
df_schema.groupBy('country') \
    .pivot('payment_method',
           ['M-Pesa', 'Card', 'Bank Transfer', 'Cash on Delivery']) \
    .agg(count('order_id')) \
    .show()

# ── Operation 6: Python UDF
print('[6/8] Python UDF — order tier classification')
@udf(returnType=StringType())
def tier_udf(total):
    if total is None:   return 'Unknown'
    if total >= 2000:   return 'Premium'
    if total >= 500:    return 'Standard'
    return 'Budget'

df_schema.withColumn('tier', tier_udf(col('order_total'))) \
    .groupBy('tier').count().orderBy('tier').show()

# ── Operation 7: Pandas UDF
print('[7/8] Pandas UDF — vectorised discount calculation')
try:
    import pyarrow  # noqa
    import pandas as pd
    import numpy as np

    @pandas_udf(DoubleType())
    def discount_udf(series: pd.Series) -> pd.Series:
        # np.where is like an if/else statement for the entire column at once
        discounted = np.where(series >= 1000, series * 0.90, series * 0.95)
        # .round(2) here belongs to Pandas, so Spark can't interfere with it
        return pd.Series(discounted).round(2)

    df_schema.withColumn('discounted_price', discount_udf(col('order_total'))) \
        .select('order_id', 'order_total', 'discounted_price') \
        .show(5)
except ImportError:
    print('   pyarrow not installed — run: pip install pyarrow')

# ── Operation 8: Write to Delta Lake on MinIO
print('[8/8] Write results to Delta Lake on MinIO')
df_high.write.format('delta').mode('overwrite') \
    .save(f'{STORAGE_ROOT}/assignment4/high_value_orders/')
print(f'   Written to {STORAGE_ROOT}/assignment4/high_value_orders/')
print('   Verify in MinIO Console -> warehouse/lab04/assignment4/')

print()
print('=' * 55)
print('All 8 required operations demonstrated.')
print('For Assignment 4: replace df_schema with your own dataset')
print('and apply these same 8 patterns.')
print('=' * 55)

Assignment 4 requires 8 operations on your own dataset.
This cell demonstrates all 8 using the lab dataset.
Adapt each section for your chosen dataset.

[1/8] Explicit schema — read from MinIO with defined schema
   Rows loaded with explicit schema: 100,000
[2/8] Filter — completed orders over $500
   High-value completed orders: 49,904
[3/8] GroupBy + Aggregation
+--------+--------+-------------+---------+
| country|n_orders|total_revenue|avg_order|
+--------+--------+-------------+---------+
|   Kenya|   34879|4.834725496E7|  1386.14|
|Tanzania|   25072| 3.44478706E7|  1373.96|
|  Uganda|   15069|2.088130141E7|  1385.71|
| Nigeria|   10051|1.420944384E7|  1413.73|
|   Ghana|    9904|1.378462945E7|  1391.82|
|Ethiopia|    5025|   7006275.15|  1394.28|
+--------+--------+-------------+---------+

[4/8] Window function — rank orders by value within country
+--------+----------+-----------+----+
| country|  order_id|order_total|rank|
+--------+----------+-----------+----+
|Ethiopia|ORD00